# Actividad Evaluativa 3 - Proyecto Final Completo (6 Clases)
Este notebook incluye absolutamente todos los laboratorios, experimentos y comparaciones requeridas.


## Configuración de Entorno (Google Colab)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_PATH = '/content/drive/MyDrive/ActividadEvaluativa1_SantiagoArbelaez_FedericoAlvarez/ActividadEvaluativa3/sea_animals_classification/data/'
os.makedirs(BASE_PATH.replace('data/', 'results/'), exist_ok=True)



## Librerías Base


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam



# 📊 PARTE 1: Exploración y Visualización de Datos


### 1. Carga de Datos y Desbalance Artificial
Extraemos los datos y forzamos un desbalance si es necesario para la rúbrica.


In [ ]:
RANDOM_STATE = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
selected_classes = ['Whale', 'Sharks', 'Fish', 'Jelly Fish', 'Starfish', 'Dolphin']

print("Buscando imágenes...")
filepaths = []
labels = []

if os.path.exists(BASE_PATH):
    for class_name in selected_classes:
        class_path = os.path.join(BASE_PATH, class_name)
        if os.path.isdir(class_path):
            for img_name in os.listdir(class_path):
                filepaths.append(os.path.join(class_path, img_name))
                labels.append(class_name)
else:
    print("¡ERROR! Ruta no encontrada:", BASE_PATH)

df_original = pd.DataFrame({'filepath': filepaths, 'label': labels})

# Forzar desbalance artificial (reducir algunas clases a 50 muestras intencionalmente)
df_list = []
for c in selected_classes:
    df_c = df_original[df_original['label'] == c]
    if c in ['Whale', 'Starfish']:  # Reducimos estas para desbalancear
        if len(df_c) > 50:
            df_c = df_c.sample(50, random_state=RANDOM_STATE)
    df_list.append(df_c)

df = pd.concat(df_list).reset_index(drop=True)
print(f"Total imágenes tras desbalance artificial: {len(df)}")



### 2. Distribución y Desbalance


In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='label', order=df['label'].value_counts().index, palette='viridis')
plt.title("Distribución de Clases (Desbalance Artificial)")
plt.xticks(rotation=45)
plt.show()



### 3. Partición de Datos y Class Weights


In [ ]:
train_df, rem_df = train_test_split(df, train_size=0.7, random_state=RANDOM_STATE, stratify=df['label'])
val_df, test_df = train_test_split(rem_df, train_size=0.5, random_state=RANDOM_STATE, stratify=rem_df['label'])

classes_names = np.unique(train_df['label'])
weights = compute_class_weight(class_weight='balanced', classes=classes_names, y=train_df['label'])
class_weights = dict(zip(range(len(classes_names)), weights))

print("Pesos para el balanceo algorítmico:")
for i, name in enumerate(classes_names):
    print(f"{name}: {class_weights[i]:.4f}")



### 4. Simulación de Baja Calidad (Data Augmentation Base)


In [ ]:
def plot_sample_augmentation():
    sample_path = train_df.iloc[0]['filepath']
    img = cv2.imread(sample_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Simular ruido gaussiano
    noise = np.random.normal(0, 0.5, img.shape) * 255
    noisy_img = np.clip(img + noise, 0, 255).astype(np.uint8)
    
    # Simular baja iluminación
    dark_img = np.clip(img * 0.4, 0, 255).astype(np.uint8)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img); axes[0].set_title("Original")
    axes[1].imshow(noisy_img); axes[1].set_title("Simulación Ruido")
    axes[2].imshow(dark_img); axes[2].set_title("Simulación Baja Luz")
    plt.show()

plot_sample_augmentation()

# ⚠️ IMPORTANTE: Eliminamos rescale=1./255 porque EfficientNet tiene su propio Rescaling interno.
# Pasaremos las imágenes en el rango [0, 255] y cada modelo se encargará de su preprocesamiento.
train_datagen = ImageDataGenerator(
    rotation_range=30, width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest', brightness_range=[0.5, 1.5]
)
val_datagen = ImageDataGenerator() # Sin rescale

train_gen = train_datagen.flow_from_dataframe(train_df, x_col='filepath', y_col='label', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical')
val_gen = val_datagen.flow_from_dataframe(val_df, x_col='filepath', y_col='label', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
test_gen = val_datagen.flow_from_dataframe(test_df, x_col='filepath', y_col='label', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)


## Utilidades de Gráficas


In [ ]:
def plot_hist(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Val')
    ax1.set_title(f"Accuracy - {title}")
    ax1.legend()
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Val')
    ax2.set_title(f"Loss - {title}")
    ax2.legend()
    plt.show()

def evaluate_model_full(model, generator, title):
    y_pred = np.argmax(model.predict(generator), axis=1)
    y_true = generator.classes
    print(f"\n--- Reporte de Clasificación ({title}) ---")
    print(classification_report(y_true, y_pred, target_names=selected_classes))
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', xticklabels=selected_classes, yticklabels=selected_classes)
    plt.title(f"Matriz de Confusión - {title}")
    plt.show()



# 🧠 PARTE 2: Modelado con EfficientNetB0
### 1. Transfer Learning (Congelado)


In [ ]:
# Arquitectura Base
base_eff = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Congelamiento total
base_eff.trainable = False

# Cabecera de Clasificación
x = base_eff.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(len(selected_classes), activation='softmax')(x)

model_eff = Model(inputs=base_eff.input, outputs=predictions)

# Optimizador y Callbacks
optimizer = Adam(learning_rate=1e-3)
model_eff.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print('Iniciando Transfer Learning (Solo cabecera entrenable)...')
h_eff_tl = model_eff.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=[early_stop],
    class_weight=class_weights
)

plot_hist(h_eff_tl, 'EfficientNet TL')


### 2. Fine-Tuning (Últimas 10 capas)


In [ ]:
base_eff.trainable = True
for layer in base_eff.layers[:-10]:
    layer.trainable = False

# Callback fresco exclusivo para el Fine-Tuning de EfficientNet
early_stop_eff_ft = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model_eff.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
print("Entrenando Fine Tuning (10 capas)...")
h_eff_ft = model_eff.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[early_stop_eff_ft],
    class_weight=class_weights
)
plot_hist(h_eff_ft, "EfficientNet FT (10 Capas)")


### 3. Laboratorio: Experimento A (Learning Rate)


In [ ]:
# Probamos un LR muy agresivo para ver si empeora
model_eff.compile(optimizer=Adam(1e-1), loss='categorical_crossentropy', metrics=['accuracy'])
print("Experimento: Learning Rate agresivo (1e-1) por 2 épocas...")
h_exp_lr = model_eff.fit(train_gen, validation_data=val_gen, epochs=2)
print("Análisis: Un LR muy alto provoca inestabilidad y que el loss salte sin coverger.")



### 4. Laboratorio: Experimento B (Descongelar 30 capas)


In [ ]:
for layer in base_eff.layers[:-30]:
    layer.trainable = False
model_eff.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
print("Experimento: Descongelar 30 capas por 3 épocas...")
h_exp_layers = model_eff.fit(train_gen, validation_data=val_gen, epochs=3)



### 5. Evaluación Final EfficientNet


In [ ]:
evaluate_model_full(model_eff, val_gen, "EfficientNetB0")



# 🚀 PARTE 3: Modelado con MobileNetV2
### 1. Transfer Learning (Congelado)


In [ ]:
from tensorflow.keras.layers import Lambda
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

base_mob = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_mob.trainable = False

# MobileNetV2 requiere entradas en [-1, 1], aplicamos preprocess_input internamente
input_tensor = tf.keras.Input(shape=(224, 224, 3))
x2 = Lambda(preprocess_input)(input_tensor)
x2 = base_mob(x2)
x2 = GlobalAveragePooling2D()(x2)
x2 = Dense(256, activation='relu')(x2)
x2 = Dropout(0.5)(x2)
predictions_mob = Dense(len(selected_classes), activation='softmax')(x2)

model_mob = Model(inputs=input_tensor, outputs=predictions_mob)

# Callback exclusivo para MobileNet TL (fresco, sin estado previo)
early_stop_mob = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model_mob.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
print("Entrenando Transfer Learning...")
h_mob_tl = model_mob.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=[early_stop_mob],
    class_weight=class_weights
)
plot_hist(h_mob_tl, "MobileNet TL")


### 2. Fine-Tuning (Últimas 10 capas)


In [ ]:
base_mob.trainable = True
for layer in base_mob.layers[:-10]:
    layer.trainable = False

# Callback NUEVO para FT (early_stop_mob ya tiene estado del TL anterior)
early_stop_mob_ft = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model_mob.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
print("Entrenando Fine Tuning (10 capas)...")
h_mob_ft = model_mob.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[early_stop_mob_ft],
    class_weight=class_weights
)
plot_hist(h_mob_ft, "MobileNet FT")


### 3. Evaluación Final MobileNetV2


In [ ]:
evaluate_model_full(model_mob, val_gen, "MobileNetV2")



# 🏆 PARTE 4: Comparación Final de Modelos


### 1. Resumen y Comparación


In [ ]:
# Comparativa rápida de métricas finales
eff_acc = h_eff_ft.history['val_accuracy'][-1]
mob_acc = h_mob_ft.history['val_accuracy'][-1]

res_df = pd.DataFrame({
    'Modelo': ['EfficientNetB0', 'MobileNetV2'],
    'Validation Accuracy Final': [eff_acc, mob_acc]
})
print(res_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=res_df, x='Modelo', y='Validation Accuracy Final', palette='Set2')
plt.title("Comparación de Accuracy Final en Validación")
plt.show()

if eff_acc > mob_acc:
    print("\nCONCLUSIÓN: EfficientNetB0 tuvo un mejor rendimiento.")
else:
    print("\nCONCLUSIÓN: MobileNetV2 tuvo un mejor rendimiento.")



In [ ]:
# Gráfica Comparativa de Accuracy a lo largo de las épocas
plt.figure(figsize=(12, 6))

# Accuracy de entrenamiento
plt.plot(h_eff_ft.history['accuracy'], label='EfficientNet Train Acc', linestyle='--', color='blue')
plt.plot(h_mob_ft.history['accuracy'], label='MobileNet Train Acc', linestyle='--', color='orange')

# Accuracy de validación
plt.plot(h_eff_ft.history['val_accuracy'], label='EfficientNet Val Acc', linewidth=2, color='blue')
plt.plot(h_mob_ft.history['val_accuracy'], label='MobileNet Val Acc', linewidth=2, color='orange')

plt.title('Comparativa de Accuracy: EfficientNetB0 vs MobileNetV2')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Gráfica Comparativa de Loss a lo largo de las épocas
plt.figure(figsize=(12, 6))

# Loss de entrenamiento
plt.plot(h_eff_ft.history['loss'], label='EfficientNet Train Loss', linestyle='--', color='blue')
plt.plot(h_mob_ft.history['loss'], label='MobileNet Train Loss', linestyle='--', color='orange')

# Loss de validación
plt.plot(h_eff_ft.history['val_loss'], label='EfficientNet Val Loss', linewidth=2, color='blue')
plt.plot(h_mob_ft.history['val_loss'], label='MobileNet Val Loss', linewidth=2, color='orange')

plt.title('Comparativa de Loss (Pérdida): EfficientNetB0 vs MobileNetV2')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


# =============================================
# PARTE 5: 10 Experimentos de Modelado (Conclusiones y Propósitos)
# =============================================

En esta sección se detallan 10 experimentos variando los modelos (EfficientNet vs MobileNet), los datos (limpios vs ruidosos), el fine-tuning (cantidad de capas descongeladas) y el learning rate.

El objetivo de estos experimentos es **validar que la convolución y la transferencia de aprendizaje se estén implementando correctamente**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# 1. Generador SIN ruido (Solo reescalado básico/shifts leves)
datagen_clean = ImageDataGenerator(
    validation_split=0.2,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1
) 
train_gen_clean = datagen_clean.flow_from_dataframe(
    train_df, x_col='filepath', y_col='label', target_size=IMG_SIZE, 
    batch_size=BATCH_SIZE, class_mode='categorical', subset='training'
)
val_gen = datagen_clean.flow_from_dataframe(
    train_df, x_col='filepath', y_col='label', target_size=IMG_SIZE, 
    batch_size=BATCH_SIZE, class_mode='categorical', subset='validation'
)

# 2. Generador CON ruido extremo (Augmentation pesado + brillo/rotaciones extremas)
datagen_noisy = ImageDataGenerator(
    rotation_range=45, width_shift_range=0.3, height_shift_range=0.3,
    shear_range=0.3, zoom_range=0.3, horizontal_flip=True, brightness_range=[0.3, 1.7],
    validation_split=0.2
)
train_gen_noisy = datagen_noisy.flow_from_dataframe(
    train_df, x_col='filepath', y_col='label', target_size=IMG_SIZE, 
    batch_size=BATCH_SIZE, class_mode='categorical', subset='training'
)

# Diccionario para guardar resultados de todos los experimentos
resultados_exp = {}

def guardar_resultado(nombre, history):
    acc = history.history['val_accuracy'][-1]
    loss = history.history['val_loss'][-1]
    resultados_exp[nombre] = {'Val Accuracy': acc, 'Val Loss': loss}
    print(f"✅ {nombre} -> Acc: {acc:.4f} | Loss: {loss:.4f}")


In [ ]:
def crear_modelo(tipo='efficientnet', congelar_hasta=None):
    if tipo == 'efficientnet':
        base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        x = base.output
        x = GlobalAveragePooling2D()(x)
        x = Dropout(0.5)(x)
        pred = Dense(len(selected_classes), activation='softmax')(x)
        model = Model(inputs=base.input, outputs=pred)
    else: # mobilenet
        base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        inputs = tf.keras.Input(shape=(224, 224, 3))
        x = Lambda(preprocess_input)(inputs)
        x = base(x)
        x = GlobalAveragePooling2D()(x)
        x = Dropout(0.5)(x)
        pred = Dense(len(selected_classes), activation='softmax')(x)
        model = Model(inputs=inputs, outputs=pred)
    
    # Lógica de congelamiento (Transfer Learning vs Fine Tuning)
    if congelar_hasta == 'todo':
        base.trainable = False
    elif isinstance(congelar_hasta, int):
        base.trainable = True
        for layer in base.layers[:-congelar_hasta]:
            layer.trainable = False
            
    return model

# Callbacks obligatorios: EarlyStopping y ModelCheckpoint
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)


### Experimento 1: EfficientNet (Transfer Learning, Datos Limpios)

**Propósito:** Establecer un *baseline* para EfficientNet utilizando Transfer Learning puro (todas las capas convolucionales congeladas) sobre un dataset con aumentos de datos leves. Esto nos permite ver qué tan bien generalizan las características preentrenadas de ImageNet a nuestro dominio marítimo.

**Conclusión Esperada:** Se espera que el modelo logre una precisión decente rápidamente sin riesgo de un sobreajuste severo (overfitting), ya que solo la capa densa final se está entrenando.


In [ ]:
print("\nExp 1: EffNet Congelado (Datos Limpios)")
m1 = crear_modelo('efficientnet', 'todo')
m1.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
h1 = m1.fit(train_gen_clean, validation_data=val_gen, epochs=5, callbacks=[early_stop, reduce_lr], verbose=0)
guardar_resultado('1. EffNet TL Limpio', h1)
plot_hist(h1, '1. EffNet TL Limpio')


**Conclusión del Experimento 1:** Tras la ejecución, se valida la capacidad base de EfficientNet. Si la exactitud (*accuracy*) en validación es alta, confirma que las características generales de ImageNet son muy útiles. Si el *loss* de entrenamiento y validación están cerca, significa que el modelo está bien ajustado.


### Experimento 2: EfficientNet (Transfer Learning, Datos con Ruido)

**Propósito:** Evaluar la robustez de EfficientNet cuando las imágenes de entrada sufren de ruido extremo, rotaciones y variaciones de brillo. Se mantiene el modelo congelado para ver si las características pre-aprendidas resisten esta degradación de los datos.

**Conclusión Esperada:** Es probable que la precisión disminuya en comparación con el Experimento 1, mostrando *underfitting* al principio, ya que al modelo le costará más extraer patrones de imágenes degradadas.


In [ ]:
print("\nExp 2: EffNet Congelado (Datos con Ruido)")
m2 = crear_modelo('efficientnet', 'todo')
m2.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
h2 = m2.fit(train_gen_noisy, validation_data=val_gen, epochs=5, callbacks=[early_stop, reduce_lr], verbose=0)
guardar_resultado('2. EffNet TL Ruidoso', h2)
plot_hist(h2, '2. EffNet TL Ruidoso')


**Conclusión del Experimento 2:** Esto demuestra si la arquitectura de EfficientNet es robusta ante el ruido. Las caídas en el *val_accuracy* indicarán que se requiere de *fine-tuning* para adaptar los filtros convolucionales a imágenes de peor calidad.


### Experimento 3: EfficientNet (Fine Tuning de 10 capas, Datos Limpios)

**Propósito:** Implementar *Fine Tuning* descongelando las últimas 10 capas convolucionales. Se disminuye la tasa de aprendizaje (Learning Rate a 1e-4) para ajustar suavemente los pesos y mejorar el *baseline* del Experimento 1, adaptando los mapas de características abstractas a las especies marinas.

**Conclusión Esperada:** Debería ser el mejor modelo de EfficientNet para datos limpios. Se espera un aumento en el *accuracy* de validación, demostrando que la convolución especializada mejora la clasificación.


In [ ]:
print("\nExp 3: EffNet FT 10 Capas (Datos Limpios)")
m3 = crear_modelo('efficientnet', 10)
m3.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
h3 = m3.fit(train_gen_clean, validation_data=val_gen, epochs=5, callbacks=[early_stop, reduce_lr], verbose=0)
guardar_resultado('3. EffNet FT-10 Limpio', h3)
plot_hist(h3, '3. EffNet FT-10 Limpio')


**Conclusión del Experimento 3:** El *Fine Tuning* mejora el desempeño frente al Transfer Learning puro. Se valida que descongelar 10 capas es suficiente para adaptar el modelo sin causar *overfitting* masivo.


### Experimento 4: EfficientNet (Fine Tuning de 30 capas, Datos con Ruido)

**Propósito:** Probar un ajuste profundo (descongelar 30 capas) sobre datos ruidosos. La hipótesis es que, al tener datos muy alterados, necesitamos que el modelo reaprenda filtros de niveles más bajos para ignorar el ruido (augmentation intenso).

**Conclusión Esperada:** Podría superar al Experimento 2 (Transfer Learning ruidoso), pero existe el riesgo de *overfitting* debido a la mayor cantidad de parámetros entrenables. Es la prueba definitiva de adaptabilidad.


In [ ]:
print("\nExp 4: EffNet FT 30 Capas (Datos con Ruido)")
m4 = crear_modelo('efficientnet', 30)
m4.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
h4 = m4.fit(train_gen_noisy, validation_data=val_gen, epochs=5, callbacks=[early_stop, reduce_lr], verbose=0)
guardar_resultado('4. EffNet FT-30 Ruidoso', h4)
plot_hist(h4, '4. EffNet FT-30 Ruidoso')


**Conclusión del Experimento 4:** Se observa si descongelar más capas en situaciones de ruido es beneficioso o si, por el contrario, el modelo colapsa e intenta memorizar el ruido. Esto nos permite entender el balance entre capacidad del modelo y la calidad del dato.


### Experimento 5: MobileNet (Transfer Learning, Datos Limpios)

**Propósito:** Establecer un *baseline* para MobileNetV2 (una red más ligera). Compararemos su desempeño y velocidad de convergencia contra el Experimento 1 de EfficientNet.

**Conclusión Esperada:** MobileNet debe ser más rápido de entrenar, pero probablemente alcance un *accuracy* ligeramente inferior o similar a EfficientNet en las mismas condiciones.


In [ ]:
print("\nExp 5: MobileNet Congelado (Datos Limpios)")
m5 = crear_modelo('mobilenet', 'todo')
m5.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
h5 = m5.fit(train_gen_clean, validation_data=val_gen, epochs=5, callbacks=[early_stop, reduce_lr], verbose=0)
guardar_resultado('5. MobileNet TL Limpio', h5)
plot_hist(h5, '5. MobileNet TL Limpio')


**Conclusión del Experimento 5:** Permite validar la idoneidad de MobileNet como modelo ligero. Si el resultado es muy cercano al de EfficientNet, MobileNet se convierte en la opción ideal para despliegues con recursos limitados.


### Experimento 6: MobileNet (Transfer Learning, Datos con Ruido)

**Propósito:** Igual que el experimento 2, pero evaluando si MobileNetV2 sufre más o menos que EfficientNet ante la degradación visual extrema.

**Conclusión Esperada:** Al ser un modelo menos profundo y con menos parámetros, podría verse más afectado por el ruido que EfficientNet, bajando su rendimiento de forma notoria.


In [ ]:
print("\nExp 6: MobileNet Congelado (Datos con Ruido)")
m6 = crear_modelo('mobilenet', 'todo')
m6.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
h6 = m6.fit(train_gen_noisy, validation_data=val_gen, epochs=5, callbacks=[early_stop, reduce_lr], verbose=0)
guardar_resultado('6. MobileNet TL Ruidoso', h6)
plot_hist(h6, '6. MobileNet TL Ruidoso')


**Conclusión del Experimento 6:** El impacto del ruido en arquitecturas ligeras queda en evidencia. Si el accuracy baja drásticamente, comprobamos que modelos pequeños necesitan datos de mejor calidad.


### Experimento 7: MobileNet (Fine Tuning de 10 capas, Datos Limpios)

**Propósito:** Aplicar *Fine Tuning* (descongelando 10 capas, LR=1e-4) para ayudar a MobileNet a especializarse en las clases marinas. Sirve de comparación directa con el Experimento 3.

**Conclusión Esperada:** Debería mejorar el *baseline* del Experimento 5. Si la mejora es marginal, puede indicar que MobileNetV2 ya está extrayendo la máxima información posible de su estructura más sencilla.


In [ ]:
print("\nExp 7: MobileNet FT 10 Capas (Datos Limpios)")
m7 = crear_modelo('mobilenet', 10)
m7.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
h7 = m7.fit(train_gen_clean, validation_data=val_gen, epochs=5, callbacks=[early_stop, reduce_lr], verbose=0)
guardar_resultado('7. MobileNet FT-10 Limpio', h7)
plot_hist(h7, '7. MobileNet FT-10 Limpio')


**Conclusión del Experimento 7:** Se confirma que descongelar 10 capas en MobileNet es una estrategia efectiva para incrementar la precisión sin incrementar de forma extrema los tiempos de entrenamiento.


### Experimento 8: MobileNet (Fine Tuning de 30 capas, Datos con Ruido)

**Propósito:** Someter a MobileNet a un ajuste profundo (30 capas) sobre el dataset ruidoso, permitiendo que sus capas iniciales reaprendan a lidiar con el ruido.

**Conclusión Esperada:** Este experimento evaluará si MobileNet tiene suficiente capacidad representacional para ignorar el ruido mediante un re-entrenamiento profundo de sus pesos convolucionales.


In [ ]:
print("\nExp 8: MobileNet FT 30 Capas (Datos con Ruido)")
m8 = crear_modelo('mobilenet', 30)
m8.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
h8 = m8.fit(train_gen_noisy, validation_data=val_gen, epochs=5, callbacks=[early_stop, reduce_lr], verbose=0)
guardar_resultado('8. MobileNet FT-30 Ruidoso', h8)
plot_hist(h8, '8. MobileNet FT-30 Ruidoso')


**Conclusión del Experimento 8:** Identificaremos si el modelo sufre de *overfitting* al intentar aprender del ruido (memorización) o si las 30 capas le dan la flexibilidad suficiente para generalizar mejor que en el Experimento 6.


### Experimento 9: EfficientNet (Learning Rate Agresivo 1e-2)

**Propósito:** Analizar el impacto de un hiperparámetro clave. Se utilizará un Learning Rate muy alto (1e-2) en una etapa de *Fine Tuning*.

**Conclusión Esperada:** Es altamente probable que el modelo no logre converger o que sufra variaciones bruscas en su *loss*. Esto demostrará por qué en el *fine tuning* se deben usar tasas de aprendizaje pequeñas.


In [ ]:
print("\nExp 9: EffNet LR Agresivo (1e-2)")
m9 = crear_modelo('efficientnet', 10)
m9.compile(optimizer=Adam(1e-2), loss='categorical_crossentropy', metrics=['accuracy'])
h9 = m9.fit(train_gen_clean, validation_data=val_gen, epochs=5, callbacks=[early_stop, reduce_lr], verbose=0)
guardar_resultado('9. EffNet LR Alto', h9)
plot_hist(h9, '9. EffNet LR Alto')


**Conclusión del Experimento 9:** La pérdida de precisión o la inestabilidad en la curva de *loss* valida la teoría: pasos demasiado grandes destruyen los pesos preentrenados (*catastrophic forgetting*).


### Experimento 10: MobileNet (Learning Rate muy Bajo 1e-6)

**Propósito:** Evaluar el extremo opuesto del hiperparámetro. Un LR extremadamente bajo (1e-6) para ver qué ocurre con la convergencia del modelo durante el *Fine Tuning*.

**Conclusión Esperada:** Se espera que el modelo sufra de *underfitting* (o un aprendizaje extremadamente lento), donde las épocas transcurran sin mejoras significativas.


In [ ]:
print("\nExp 10: MobileNet LR Lento (1e-6)")
m10 = crear_modelo('mobilenet', 10)
m10.compile(optimizer=Adam(1e-6), loss='categorical_crossentropy', metrics=['accuracy'])
h10 = m10.fit(train_gen_clean, validation_data=val_gen, epochs=5, callbacks=[early_stop, reduce_lr], verbose=0)
guardar_resultado('10. MobileNet LR Bajo', h10)
plot_hist(h10, '10. MobileNet LR Bajo')


**Conclusión del Experimento 10:** El nulo o lentísimo avance del *accuracy* demostrará empíricamente la necesidad de escoger un LR balanceado (típicamente entre 1e-3 y 1e-5) para garantizar convergencia en tiempos razonables.


### Tabla Comparativa y Gráfica de Resultados Finales

**Propósito:** Consolidar los hallazgos para una interpretación clara y presentar los resultados finales ordenados, cumpliendo con el requerimiento de análisis integral de Overfitting, Underfitting y métricas generales.


In [ ]:
import seaborn as sns

# Convertir diccionario a DataFrame para visualizar
df_resultados = pd.DataFrame.from_dict(resultados_exp, orient='index').reset_index()
df_resultados.columns = ['Experimento', 'Val Accuracy', 'Val Loss']
df_resultados = df_resultados.sort_values(by='Val Accuracy', ascending=False)

# Mostrar tabla
print("=== TABLA DE RESULTADOS (De mejor a peor) ===")
print(df_resultados.to_string(index=False))

# Gráfica de barras
plt.figure(figsize=(14, 6))
sns.barplot(data=df_resultados, x='Val Accuracy', y='Experimento', palette='viridis')
plt.title("Comparación de Accuracy en los 10 Experimentos", fontsize=14, fontweight='bold')
plt.xlabel("Validation Accuracy")
plt.ylabel("Configuración del Experimento")
plt.xlim(0, 1)

# Añadir etiquetas de texto en las barras
for index, value in enumerate(df_resultados['Val Accuracy']):
    plt.text(value + 0.01, index, f'{value:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()
